# 03 · 선호 데이터와 DPO 손실

Direct Preference Optimization(DPO, 직접 선호 최적화)의 입력과 손실을 수치로 확인합니다. 단일 질문·두 고정 답변의 장난감 정책 학습만 실행합니다. 실제 언어 모델 미세조정이나 일반화 검증이 아닙니다. 약 15분.

모델 다운로드·API 키·GPU 없이 실행합니다. 합성 자료만 사용하며, 실제 LLM 학습·성능 검증과 구분합니다. 예측 → 실행 → 변수 변경 → 근거 기록의 순서로 진행합니다.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from course_labs.adaptation import *
OUT = ROOT / 'output' / 'validation' / 'adaptation-labs'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'NumPy:', np.__version__)
assert sys.version_info[:2] == (3, 11), '공통 환경 Python 3.11 커널을 선택하세요.'


## 선호쌍과 선택 이유

chosen/rejected는 절대 정답/오답이 아니라 명시한 기준에 따른 상대 선택입니다.

In [ ]:
prefs = [{'prompt':'근거 문서가 없는 문의', 'chosen':'확인할 수 없어 담당자 검토가 필요합니다.', 'rejected':'무조건 승인됩니다.', 'reason':'근거 없는 단정 방지'}]
display(prefs)
assert not preference_audit(prefs)

## 무효 쌍

동일한 두 답변은 비교 신호를 주지 않습니다.

In [ ]:
invalid = [dict(prefs[0], rejected=prefs[0]['chosen'])]
display(preference_audit(invalid))

## 길이 편향 점검

길이 차이는 검토 신호이지 선호쌍 삭제의 자동 기준이 아닙니다.

In [ ]:
for p in prefs:
    print('문자 수:', len(p['chosen']), len(p['rejected']), '선택 근거:', p['reason'])

## log probability의 의미

이 예시는 이미 계산된 시퀀스 log probability를 사용합니다. Judge 점수를 넣지 않습니다.

In [ ]:
policy = (-2.0, -5.0); reference = (-3.0, -3.0)
margin = (policy[0]-policy[1])-(reference[0]-reference[1])
print('reference 대비 선호 margin:', margin)

## 기준선의 손실

policy와 reference의 선호 차이가 같으면 log(2)입니다.

In [ ]:
loss0 = dpo_loss(-3,-3,-3,-3,beta=.1)
print(loss0)
assert np.isclose(loss0,np.log(2))

## 선호 방향의 변화

chosen의 상대 log probability가 증가할 때 해당 쌍의 손실이 줄어듭니다.

In [ ]:
good = dpo_loss(-2,-5,-3,-3,.1)
bad = dpo_loss(-5,-2,-3,-3,.1)
print('선호 방향 / 반대 방향:',good,bad)
assert good < loss0 < bad

## beta 민감도

하나의 margin에서 손실이 달라지는 관찰입니다. beta가 클수록 품질이 좋아진다는 결론은 아닙니다.

In [ ]:
betas = [.1,.5,1.]
display([{'beta': b, 'loss': dpo_loss(-2,-5,-3,-3,b)} for b in betas])

## reference의 영향

reference가 이미 chosen을 더 선호하면 추가 변화의 margin은 달라집니다.

In [ ]:
display([{'ref_chosen': c, 'loss': dpo_loss(-2,-5,c,-5,.1)} for c in (-5,-3,-2)])

## 수치 안정성

logaddexp를 사용해 매우 큰 margin에서 overflow를 피합니다.

In [ ]:
assert np.isfinite(dpo_loss(-10000,0,0,0))
print(dpo_loss(-10000,0,0,0))

## 두 답변 정책의 최적화

Bernoulli policy의 θ 하나를 갱신합니다. reference는 0.5/0.5로 고정합니다.

In [ ]:
history = toy_preference_training(beta=.5,steps=80,lr=.3)
display(history[::20])
assert history[-1]['loss'] < history[0]['loss']

## beta별 고정 step 실험

같은 학습률·step의 장난감 예시입니다. 실제 LLM 결과로 승격하지 마세요.

In [ ]:
ab = []
for b in betas:
    run=toy_preference_training(beta=b)
    ab.append({'beta':b, **run[-1]})
display(ab)

## 선호 확률 시각화

한 쌍을 더 선호하게 되었을 뿐, 새로운 질문의 안전성과 정확성을 입증하지 않습니다.

In [ ]:
display(HTML(''.join(f"<p>step {r['step']:02} <span style='display:inline-block;background:#284b72;height:14px;width:{r['chosen_probability']*400}px'></span> {r['chosen_probability']:.3f}</p>" for r in history[::20])))

## 평가표의 분리

실제 모델에서는 동일 입력의 유용성·근거·형식·과도한 거절을 따로 확인합니다.

In [ ]:
evaluation_plan = {'fixed_prompts':10, 'dimensions':['유용성','근거','형식','과도한 거절'], 'baseline':'SFT model', 'blind_order':True, 'judge':'human review + explicit rubric', 'limitation':'모델 비교 미실행'}
save_report(OUT / 'preference-report.json', {'toy_runs':ab,'evaluation_plan':evaluation_plan})

## 실행 기록과 기획서 연결

1. 바꾼 변수와 달라진 결과
2. 이 실험으로 확인하지 못한 부분
3. 프로젝트에서 추가로 검증할 가정

출력은 `output/validation/adaptation-labs/`에 저장됩니다. 개인정보나 학생 제출물은 공개 저장소에 넣지 않습니다.